# For cleaning `internships` and `jobs

In [31]:
import pandas as pd


# dataframes to parquet
def save_parquet(df, path, index=False):
    df.to_parquet(path, index=index)


# load raw data
jobs = pd.read_csv("../data/raw/Fake_Job_Posting_Detection_Dataset.csv")
internships = pd.read_csv("../data/raw/fake_internship_detection_dataset.csv")

# all lowercase
jobs.columns = jobs.columns.str.lower()


# unique column names
print(set(jobs.columns) - set(internships.columns))
print(set(internships.columns) - set(jobs.columns))

{'label', 'company_type', 'job_title', 'salary_inr'}
{'is_fake_posting', 'company_age', 'grammatical_errors', 'posting_date', 'website_available', 'vague_description_score', 'verification_status', 'recruiter_response_time_hours', 'work_mode', 'keyword_spam_score', 'stipend', 'registration_fee', 'suspicious_email_domain', 'company_size', 'urgency_score', 'social_media_presence', 'industry', 'payment_required', 'company_name', 'location', 'fake_certificate_offer', 'emotional_manipulation_score', 'recruiter_email_type', 'linkedin_presence', 'fraud_score', 'unrealistic_salary_flag', 'internship_title', 'job_description_length', 'trust_signal_score', 'phishing_language_score', 'recruiter_experience_years', 'domain_age_months'}


## columns unique to `jobs`
 job_title, company_type, label, salary_inr

## columns unique to `internships`
 'company_name', 'posting_date', 'trust_signal_score', 'job_description_length',
 'recruiter_email_type', 'recruiter_experience_years', 'fake_certificate_offer',
 'emotional_manipulation_score', 'is_fake_posting', 'work_mode', 'website_available',
 'registration_fee', 'grammatical_errors', 'fraud_score', 'location', 'phishing_language_score',
 'urgency_score', 'keyword_spam_score', 'payment_required', 'suspicious_email_domain',
 'internship_title', 'unrealistic_salary_flag', 'company_age', 'domain_age_months',
 'recruiter_response_time_hours', 'linkedin_presence', 'industry', 'company_size',
 'social_media_presence', 'verification_status', 'vague_description_score', 'stipend'

special note: `job_title` from `jobs` and `internship_title` from `internships` are probably the same

In [32]:
# rename `internship_title` to `job_title`
internships.rename(columns={"internship_title": "job_title"}, inplace=True)

# save to parquet
internships.to_parquet("../data/clean/internships.parquet")
jobs.to_parquet("../data/clean/jobs.parquet")

C:\Users\laura\AppData\Local\Temp\ipykernel_2452\1104830884.py:2: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  internships.rename(columns={"internship_title": "job_title"}, inplace=True)


outer merge to make one massive table

In [33]:
full = pd.concat(
    [internships.assign(source="internship"), jobs.assign(source="job")],
    ignore_index=True,
)

In [34]:
print(jobs.shape, internships.shape, full.shape)

(2000, 5) (1000000, 33) (1002000, 37)


In [35]:
full = full.reset_index(drop=True)
full.insert(0, "posting_id", full.index + 1)

Table 1: Company details

This contains all columns pertaining to company details

- `industry`
- `location`
- `company_name`
- `company_size`
- `company_age`
- `linkedin_presence`
- `website_available`
- `dmoain_age_months`


In [ ]:
company_details = full.loc[
    :,
    [
        "posting_id",
        "industry",
        "location",
        "company_name",
        "company_size",
        "company_age",
        "linkedin_presence",
        "website_available",
        "domain_age_months",
    ],
]

Table 2: Recruiter details

This table contains the company's name and indicators of online presence

- `company_name`
- `recruiter_experience_years`
- `recruiter_email_type`
- `suspicious_email_domain`
- `recruiter_response_time_hours`
- `social_media_presence` (recruiter social media presence)

In [40]:
recruiters = full.loc[
    :,
    [
        "posting_id",
        "company_name",
        "recruiter_experience_years",
        "recruiter_email_type",
        "suspicious_email_domain",
        "recruiter_response_time_hours",
        "social_media_presence",
    ],
]

Table 3: Posting details

- `posting_date`
- `job_title`
- `employment_type`
- `work_mode`
- `industry`
- `company_name`
- `stipend` (I will probably change this name to something more intuitive later)
- `job_description_length`

In [41]:
postings = full.loc[
    :,
    [
        "posting_id",
        "posting_date",
        "job_title",
        "employment_type",
        "work_mode",
        "industry",
        "company_name",
        "stipend",
        "job_description_length",
    ],
]

Table 4: Sentiment scores

- `posting_date`
- `internship_title`
- `company_name`
- `verification_status`
- `unrealistic_salary_flag`
- `payment_required`
- `registration_fee`
- `grammatical_errors`
- `vague_description_score`
- `urgency_score`
- `keyword_spam_score`
- `fake_certificate_offer`
- `emotional_manipulation_score`
- `phishing_language_score`
- `trust_signal_score`
- `fraud_score`
- `is_fake_posting`

In [43]:
fraud_signals = full.loc[
    :,
    [
        "posting_id",
        "posting_date",
        "job_title",
        "company_name",
        "verification_status",
        "unrealistic_salary_flag",
        "payment_required",
        "registration_fee",
        "grammatical_errors",
        "vague_description_score",
        "urgency_score",
        "keyword_spam_score",
        "fake_certificate_offer",
        "emotional_manipulation_score",
        "phishing_language_score",
        "trust_signal_score",
    ],
]

## Export subsetted tables as parquet files

In [44]:
tables = {
    "company_details": company_details,
    "postings": postings,
    "recruiters": recruiters,
    "fraud_signals": fraud_signals,
}

for name, df in tables.items():
    save_parquet(df, f"../data/clean/{name}.parquet")